# Verifying Amplitudes and Correlation Energies

TODO:
- Create a pandas dataframe with the following structure:

structure | basis set | PySCF_Corr | Psi4_Corr | PySCF_norm_t1 | Psi4_norm_t1 | PySCF_norm_t2 | Psi4_norm_t2

- Populate the data frame with the provided code
- Use a try-except to handle linalg error, save a list of faulty structures
- Calculate the mean error for each structure basis combo


In [ ]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm


from glob import glob
import psi4
from helper_CC_ML_spacial import *

import pyscf
import pyscf.cc
import pyscf.mcscf


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from numpy.linalg import LinAlgError
import psi4
import pyscf
from pyscf import gto, scf, cc

# Import HelperCCEnergy for Psi4

# Path to XYZ files
xyz_folder = 'diatomics'
xyz_files = glob.glob(os.path.join(xyz_folder, '*.xyz'))

# Extract structure names and contents
structures = []
structure_contents = {}

for file_path in xyz_files:
    structure_name = os.path.splitext(os.path.basename(file_path))[0]
    structures.append(structure_name)
    with open(file_path, 'r') as f:
        structure_contents[structure_name] = f.read()

# Basis sets to iterate over

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

results = []
failures = []

for structure in structures:
    xyz_path = os.path.join(xyz_folder, f'{structure}.xyz')
    with open(xyz_path, 'r') as f:
        xyz_text = f.read()

    for basis in basis_sets:
        # Initialize default values
        psi4_corr = np.nan
        pyscf_corr = np.nan
        norm_t1_psi4 = np.nan
        norm_t2_psi4 = np.nan
        norm_t1_pyscf = np.nan
        norm_t2_pyscf = np.nan

        ##### PSI4 Processing #####
        try:
            qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
            mol_psi4 = psi4.geometry(qmol.create_psi4_string_from_molecule() + 'symmetry c1')

            psi4.core.clean()
            psi4.core.be_quiet()

            psi4.set_options({
                'basis': basis,
                'scf_type': 'pk',
                'reference': 'rhf',
                'mp2_type': 'conv',
                'e_convergence': 1e-8,
                'd_convergence': 1e-8
            })

            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            A = HelperCCEnergy(mol_psi4, rhf_e, scf_wfn, freeze_core=False)

            try:
                A.compute_energy()
                psi4_corr = A.FinalEnergy
            except LinAlgError:
                failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Corr'})

            # Extract amplitudes
            norm_t1_psi4 = np.linalg.norm(A.t1)
            norm_t2_psi4 = np.linalg.norm(A.t1)

        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Setup', 'error': str(e)})

        ##### PYSCF Processing #####
        try:
            mol_pyscf = gto.Mole()
            mol_pyscf.build(atom=xyz_path, basis=basis, symmetry='c1')

            # Define active space (assuming no frozen orbitals for simplicity)
            n_frozen = 0
            active_space = range(n_frozen, mol_pyscf.nao_nr())
            scf_calc = scf.RHF(mol_pyscf).run()
            

            # Compute CCSD
            ccsd_calc = cc.CCSD(scf_calc, frozen=[i for i in range(mol_pyscf.nao_nr()) if i not in active_space]).run()
            pyscf_corr = ccsd_calc.e_corr

            t1_pyscf = ccsd_calc.t1
            t2_pyscf = ccsd_calc.t2
            norm_t1_pyscf = np.linalg.norm(t1_pyscf)
            norm_t2_pyscf = np.linalg.norm(t2_pyscf)

        except LinAlgError:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Corr'})
        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Setup', 'error': str(e)})

        # Store results
        results.append({
            'structure': structure,
            'basis set': basis,
            'PySCF_Corr': pyscf_corr,
            'Psi4_Corr': psi4_corr,
            'PySCF_norm_t1': norm_t1_pyscf,
            'Psi4_norm_t1': norm_t1_psi4,
            'PySCF_norm_t2': norm_t2_pyscf,
            'Psi4_norm_t2': norm_t2_psi4
        })

# Create and display final DataFrame
df = pd.DataFrame(results)
print("\nFinal DataFrame:")
print(df)

if failures:
    print("\nFailures Encountered:")
    for failure in failures:
        print(f"Structure: {failure['structure']}, Basis Set: {failure['basis set']}, Method: {failure['method']}, Error: {failure.get('error', '')}")

# Optional: Save results to CSV
# df.to_csv('amplitude_correlation_results.csv', index=False)


In [ ]:
failures_df =pd.DataFrame(failures)

In [ ]:
df

In [ ]:
df.to_csv('results.csv')


In [ ]:
df.loc[df['structure'] == 'BB']